# 13. The Final Scrub: Complete Data Cleaning Workflow

## Congratulations!

You've made it to the end of the data cleaning module! You now have the fundamental skills to take messy, real-world raw data and transform it into clean, analysis-ready DataFrames.

---

## Quick Recap: The 4 Main Data Cleaning Pillars

Before taking on the final project, here is a quick cheat sheet summarizing the four primary data quality issues and how to solve them in Pandas:

### 1. Missing Data

Incomplete rows can throw off calculations or break machine learning models.

```python
# Remove any row with missing data
cleaned_df = df.dropna()

# Remove rows with missing data in specific columns
cleaned_df = df.dropna(subset=['username', 'location'])

# Replace missing string values
df['location'] = df['location'].fillna('Unknown')

# Impute missing numerical values with the mean
mean_age = df['age'].mean()
df['age'] = df['age'].fillna(mean_age)

```

---

### 2. Duplicate Data

Repeated entries skew totals, averages, and statistics.

```python
# Identify duplicate rows (returns True/False series)
df.duplicated()

# Filter and view duplicate rows
df[df.duplicated()]

# Remove exact duplicate rows across all columns
no_dupes = df.drop_duplicates()

# Remove duplicate entries based on a unique identifier
unique_records = df.drop_duplicates(subset=['user_id'])

```

---

### 3. Wrong Data Types

Operations fail when numbers or booleans are incorrectly stored as text.

```python
# Inspect current data types across all columns
df.dtypes

# Convert text column to numeric (coercing errors to NaN)
df['age'] = pd.to_numeric(df['age'], errors='coerce')

```

---

### 4. Inconsistent Strings

User-entered text varies in casing, spacing, and phrasing.

```python
# Standardize text: convert to lower case and strip whitespace
responses['rsvp'] = responses['rsvp'].str.lower().str.strip()

# Replace specific substrings or values
responses['rsvp'] = responses['rsvp'].str.replace('yes', 'y')

```

---

## Final Project: Emergency Department Scrub

We've practiced each of these data cleaning tasks in isolation. In the real world, you'll apply all of these strategies together on a single dataset.

> **The Scenario:** Hospital emergency rooms (*inspired by shows like Grey's Anatomy, House, and The Pitt*) operate under fast-paced, high-pressure conditions. Because data is entered quickly and under stress, intake records are notoriously messy—full of missing values, duplicate entries, mismatched text casing, and incorrect data types.

You have been provided with an **Emergency Department Patient Intake DataFrame** that requires a full scrub before the medical director can analyze hospital efficiency.

---

### Suggested Cleaning Steps

There is no single "correct" sequence, but a standard cleaning pipeline often follows this flow:

```
[1. Inspect]       Check df.info(), df.dtypes, and df.head()
      ↓
[2. Deduplicate]   Remove duplicate logging errors with .drop_duplicates()
      ↓
[3. Standardize]   Clean string casing & trailing spaces with .str methods
      ↓
[4. Type Cast]     Convert numeric fields using pd.to_numeric(errors='coerce')
      ↓
[5. Handle NaNs]   Fill or drop missing values based on clinical context

```

### Your Tasks

1. **Inspect:** Examine the intake dataset to identify missing data, duplicates, and bad types.
2. **Deduplicate:** Remove redundant records representing system logging errors.
3. **Normalize Text:** Standardize medical priority categories (e.g., convert `'critical'`, `'CRITICAL'`, and `'Critical'` into a uniform `'critical'`).
4. **Fix Types:** Convert numerical intake measurements (e.g., heart rate, wait time) into numeric data types.
5. **Handle Missing Values:** Impute or drop empty fields based on clinical relevance.

In [1]:
import pandas as pd

data = {
    "patient_id": [
        "PT-2001", "PT-2001", "PT-2002", "PT-2003",
        "PT-2004", "PT-2005", "PT-2006", "PT-2006",
        "PT-2007", "PT-2008", "PT-2009", "PT-2010",
        "PT-2011",
    ],
    "name": [
        "Anthony Ramirez", "anthony ramirez", "Grace Mitchell",
        "Daniel Brooks", "Nina Patel", "Marcus Johnson",
        "Elena Cruz", "elena cruz", "William Carter",
        "Sophia Nguyen", None, "Sophia Nguyen", "Ethan Walker",
    ],
    "age": [
        "52", "52", 34, "28", "61", 47, "26",
        26, "73", "39", 44, "39", "58",
    ],
    "department": [
        "ER", "er", "Trauma", "ER", "Cardiology", "Trauma",
        "ER", "er", "ICU", "ER", "ER", "ER", "ICU",
    ],
    "triage_level": [
        "Critical", "critical", "Moderate", "HIGH", "Low",
        "Moderate", "critical", "CRITICAL", "High", "Low",
        "Moderate", "low", None,
    ],
    "wait_time_minutes": [
        "12", "12", 55, "40 mins", 140, 60, "8",
        8, "95", "25", None, "25", "70",
    ],
    "admitted": [
        True, True, False, True, True, False, True,
        True, False, True, False, True, None,
    ],
}

patients = pd.DataFrame(data)
patients

,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,ER,Critical,12,True
1,PT-2001,anthony ramirez,52,er,critical,12,True
2,PT-2002,Grace Mitchell,34,Trauma,Moderate,55,False
3,PT-2003,Daniel Brooks,28,ER,HIGH,40 mins,True
4,PT-2004,Nina Patel,61,Cardiology,Low,140,True
5,PT-2005,Marcus Johnson,47,Trauma,Moderate,60,False
6,PT-2006,Elena Cruz,26,ER,critical,8,True
7,PT-2006,elena cruz,26,er,CRITICAL,8,True
8,PT-2007,William Carter,73,ICU,High,95,False
9,PT-2008,Sophia Nguyen,39,ER,Low,25,True


## 1. Inspect the original data

We inspect:

- the first rows;
- column data types;
- missing-value counts;
- repeated patient IDs.

At this stage, `drop_duplicates()` alone would not remove the duplicated logging records because values such as `"ER"` and `"er"` are still different strings.


In [ ]:
print("First five rows:")
patients.head()

print("\nDataFrame information:")
patients.info()

print("\nColumn data types:")
patients.dtypes.to_frame(name="dtype")

print("\nMissing values per column:")
patients.isna().sum().to_frame(name="missing_values")

print("\nRows with repeated patient IDs:")
repeated_patient_ids = patients[
    patients.duplicated(subset=["patient_id"], keep=False)
]
repeated_patient_ids

print(f"\nExact duplicate rows before standardization: {patients.duplicated().sum()}")

First five rows:


,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,ER,Critical,12,True
1,PT-2001,anthony ramirez,52,er,critical,12,True
2,PT-2002,Grace Mitchell,34,Trauma,Moderate,55,False
3,PT-2003,Daniel Brooks,28,ER,HIGH,40 mins,True
4,PT-2004,Nina Patel,61,Cardiology,Low,140,True



DataFrame information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   patient_id         13 non-null     object
 1   name               12 non-null     object
 2   age                13 non-null     object
 3   department         13 non-null     object
 4   triage_level       12 non-null     object
 5   wait_time_minutes  12 non-null     object
 6   admitted           12 non-null     object
dtypes: object(7)
memory usage: 860.0+ bytes

Column data types:


,dtype
patient_id,object
name,object
age,object
department,object
triage_level,object
wait_time_minutes,object
admitted,object



Missing values per column:


,missing_values
patient_id,0
name,1
age,0
department,0
triage_level,1
wait_time_minutes,1
admitted,1



Rows with repeated patient IDs:


,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,ER,Critical,12,True
1,PT-2001,anthony ramirez,52,er,critical,12,True
6,PT-2006,Elena Cruz,26,ER,critical,8,True
7,PT-2006,elena cruz,26,er,CRITICAL,8,True



Exact duplicate rows before standardization: 0


## 2. Create a cleaning copy

The original DataFrame is preserved. All cleaning operations are performed on `patients_clean`.


In [3]:
patients_clean = patients.copy()

## 3. Standardize text

- Patient IDs are stripped and converted to uppercase.
- Names are stripped and converted to title case.
- Departments are stripped and converted to uppercase.
- Triage levels are stripped and converted to lowercase.

Pandas' nullable `string` type preserves missing values as `<NA>`.


In [4]:
patients_clean["patient_id"] = (
    patients_clean["patient_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

patients_clean["name"] = (
    patients_clean["name"]
    .astype("string")
    .str.strip()
    .str.title()
)

patients_clean["department"] = (
    patients_clean["department"]
    .astype("string")
    .str.strip()
    .str.upper()
)

patients_clean["triage_level"] = (
    patients_clean["triage_level"]
    .astype("string")
    .str.strip()
    .str.lower()
)

patients_clean

,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,ER,critical,12,True
1,PT-2001,Anthony Ramirez,52,ER,critical,12,True
2,PT-2002,Grace Mitchell,34,TRAUMA,moderate,55,False
3,PT-2003,Daniel Brooks,28,ER,high,40 mins,True
4,PT-2004,Nina Patel,61,CARDIOLOGY,low,140,True
5,PT-2005,Marcus Johnson,47,TRAUMA,moderate,60,False
6,PT-2006,Elena Cruz,26,ER,critical,8,True
7,PT-2006,Elena Cruz,26,ER,critical,8,True
8,PT-2007,William Carter,73,ICU,high,95,False
9,PT-2008,Sophia Nguyen,39,ER,low,25,True


## 4. Fix data types

The `age` column is converted to nullable integers.

Before converting wait times, the text `" mins"` is removed. This preserves the valid value in `"40 mins"` instead of converting it to missing data.

The `admitted` column is converted to Pandas' nullable Boolean type, which supports `True`, `False`, and `<NA>`.


In [5]:
patients_clean["age"] = (
    pd.to_numeric(
        patients_clean["age"],
        errors="coerce",
    )
    .astype("Int64")
)

patients_clean["wait_time_minutes"] = (
    patients_clean["wait_time_minutes"]
    .astype("string")
    .str.replace(" mins", "", regex=False)
    .str.strip()
)

patients_clean["wait_time_minutes"] = (
    pd.to_numeric(
        patients_clean["wait_time_minutes"],
        errors="coerce",
    )
    .astype("Float64")
)

patients_clean["admitted"] = (
    patients_clean["admitted"]
    .astype("boolean")
)

print("Data types after conversion:")
display(patients_clean.dtypes.to_frame(name="dtype"))

print("\nDaniel Brooks' preserved wait time:")
display(
    patients_clean.loc[
        patients_clean["patient_id"].eq("PT-2003"),
        ["patient_id", "name", "wait_time_minutes"],
    ]
)

Data types after conversion:


,dtype
patient_id,string[python]
name,string[python]
age,Int64
department,string[python]
triage_level,string[python]
wait_time_minutes,Float64
admitted,boolean



Daniel Brooks' preserved wait time:


,patient_id,name,wait_time_minutes
3,PT-2003,Daniel Brooks,40.0


## 5. Remove duplicate logging records

After standardization and type conversion, the two logging duplicates become exact duplicate rows.

For this exercise, `drop_duplicates()` across all columns is safer than assuming every repeated `patient_id` is automatically invalid. In a real hospital system, one patient could have multiple legitimate visits.


In [6]:
rows_before = len(patients_clean)

duplicate_rows = patients_clean[
    patients_clean.duplicated(keep=False)
]

print("Exact duplicate rows after standardization:")
display(duplicate_rows)

patients_clean = (
    patients_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

rows_removed = rows_before - len(patients_clean)

print(f"Rows before deduplication: {rows_before}")
print(f"Rows removed: {rows_removed}")
print(f"Rows remaining: {len(patients_clean)}")

Exact duplicate rows after standardization:


,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,ER,critical,12.0,True
1,PT-2001,Anthony Ramirez,52,ER,critical,12.0,True
6,PT-2006,Elena Cruz,26,ER,critical,8.0,True
7,PT-2006,Elena Cruz,26,ER,critical,8.0,True


Rows before deduplication: 13
Rows removed: 2
Rows remaining: 11


## 6. Handle missing values

### Name

A missing name does not make the rest of the record useless because the patient still has an ID. It is replaced with `"Unknown"`.

### Triage level

A medical priority should not be guessed from the most common category. A missing value is labelled `"unknown"`.

### Wait time

The median is used because it is less affected by unusually high values such as `140` minutes. The `wait_time_imputed` column records which value was estimated.

### Admission status

The unknown admission status remains `<NA>`. Replacing it with `True` or `False` would create an unsupported medical fact.


In [7]:
patients_clean["name"] = (
    patients_clean["name"]
    .fillna("Unknown")
)

patients_clean["triage_level"] = (
    patients_clean["triage_level"]
    .fillna("unknown")
)

patients_clean["wait_time_imputed"] = (
    patients_clean["wait_time_minutes"]
    .isna()
)

median_wait_time = patients_clean["wait_time_minutes"].median()

patients_clean["wait_time_minutes"] = (
    patients_clean["wait_time_minutes"]
    .fillna(median_wait_time)
)

print(f"Median wait time used for imputation: {median_wait_time} minutes")
patients_clean

Median wait time used for imputation: 47.5 minutes


,patient_id,name,age,department,triage_level,wait_time_minutes,admitted,wait_time_imputed
0,PT-2001,Anthony Ramirez,52,ER,critical,12.0,True,False
1,PT-2002,Grace Mitchell,34,TRAUMA,moderate,55.0,False,False
2,PT-2003,Daniel Brooks,28,ER,high,40.0,True,False
3,PT-2004,Nina Patel,61,CARDIOLOGY,low,140.0,True,False
4,PT-2005,Marcus Johnson,47,TRAUMA,moderate,60.0,False,False
5,PT-2006,Elena Cruz,26,ER,critical,8.0,True,False
6,PT-2007,William Carter,73,ICU,high,95.0,False,False
7,PT-2008,Sophia Nguyen,39,ER,low,25.0,True,False
8,PT-2009,Unknown,44,ER,moderate,47.5,False,True
9,PT-2010,Sophia Nguyen,39,ER,low,25.0,True,False


## 7. Validate the cleaned DataFrame

The assertions below verify the most important cleaning decisions. If an assertion fails, Python raises an error and identifies that the cleaning result is not as expected.


In [8]:
# The two duplicate logging records were removed.
assert len(patients_clean) == 11
assert patients_clean.duplicated().sum() == 0

# Numeric columns have numeric dtypes.
assert str(patients_clean["age"].dtype) == "Int64"
assert str(patients_clean["wait_time_minutes"].dtype) == "Float64"

# "40 mins" was correctly preserved as 40.
pt_2003_wait = patients_clean.loc[
    patients_clean["patient_id"].eq("PT-2003"),
    "wait_time_minutes",
].iloc[0]
assert pt_2003_wait == 40

# The record with a missing name was kept.
pt_2009 = patients_clean.loc[
    patients_clean["patient_id"].eq("PT-2009")
].iloc[0]
assert pt_2009["name"] == "Unknown"
assert bool(pt_2009["wait_time_imputed"]) is True
assert pt_2009["wait_time_minutes"] == median_wait_time

# Unknown medical statuses were not guessed.
pt_2011 = patients_clean.loc[
    patients_clean["patient_id"].eq("PT-2011")
].iloc[0]
assert pt_2011["triage_level"] == "unknown"
assert pd.isna(pt_2011["admitted"])

print("All validation checks passed.")

All validation checks passed.


## 8. Final cleaned result

The final DataFrame contains 11 unique patient records.

The only remaining missing value is the intentionally unknown admission status for `PT-2011`.


In [9]:
display(patients_clean)

print("\nFinal data types:")
display(patients_clean.dtypes.to_frame(name="dtype"))

print("\nFinal missing-value counts:")
display(patients_clean.isna().sum().to_frame(name="missing_values"))

print("\nCleaning summary:")
print(f"Original rows: {len(patients)}")
print(f"Cleaned rows: {len(patients_clean)}")
print(f"Duplicate logging rows removed: {len(patients) - len(patients_clean)}")
print(f"Wait times imputed: {patients_clean['wait_time_imputed'].sum()}")

,patient_id,name,age,department,triage_level,wait_time_minutes,admitted,wait_time_imputed
0,PT-2001,Anthony Ramirez,52,ER,critical,12.0,True,False
1,PT-2002,Grace Mitchell,34,TRAUMA,moderate,55.0,False,False
2,PT-2003,Daniel Brooks,28,ER,high,40.0,True,False
3,PT-2004,Nina Patel,61,CARDIOLOGY,low,140.0,True,False
4,PT-2005,Marcus Johnson,47,TRAUMA,moderate,60.0,False,False
5,PT-2006,Elena Cruz,26,ER,critical,8.0,True,False
6,PT-2007,William Carter,73,ICU,high,95.0,False,False
7,PT-2008,Sophia Nguyen,39,ER,low,25.0,True,False
8,PT-2009,Unknown,44,ER,moderate,47.5,False,True
9,PT-2010,Sophia Nguyen,39,ER,low,25.0,True,False



Final data types:


,dtype
patient_id,string[python]
name,string[python]
age,Int64
department,string[python]
triage_level,string[python]
wait_time_minutes,Float64
admitted,boolean
wait_time_imputed,bool



Final missing-value counts:


,missing_values
patient_id,0
name,0
age,0
department,0
triage_level,0
wait_time_minutes,0
admitted,1
wait_time_imputed,0



Cleaning summary:
Original rows: 13
Cleaned rows: 11
Duplicate logging rows removed: 2
Wait times imputed: 1
